In [ ]:
PROJECT_RANDOM_STATE = 2627
print("PROJECT_RANDOM_STATE =", PROJECT_RANDOM_STATE)

# Practice 4 — Challenge A (cyber): decision tree and complexity control

**Contents:** P4.1, P4.2, P4.4 (feature importances) and the Challenge A part of P4.5. The two rules of P4.4 are presented in `p4_storage.ipynb`, because they come from the Challenge B tree.

## 1. Setup

In [ ]:
import json
import time
import platform
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, validation_curve
from sklearn import metrics

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib:", matplotlib.__version__)

CYBER_DIR = "../datos/cyber"
FOLDS_DIR = "../folds"

## 2. Data

In [ ]:
# Load the tables and the class codes without modifying the original files.
dev = pd.read_csv(CYBER_DIR + "/cyber_development.csv")
cyber_groups = pd.read_csv(CYBER_DIR + "/cyber_groups.csv")
cyber_folds = pd.read_csv(FOLDS_DIR + "/cyber_folds.csv")

# The JSON keys are text; convert them to the integer codes used in y.
with open(CYBER_DIR + "/cyber_class_codes.json", encoding="utf-8") as file:
    class_codes = json.load(file)

CLASS_NAMES = {}
for code, name in class_codes.items():
    CLASS_NAMES[int(code)] = name

# The 11 source predictors of P2. Metadata columns are never predictors.
NOMINAL = ["proto", "service", "conn_state"]
NUMERIC = ["duration", "src_bytes", "dst_bytes", "missed_bytes",
           "src_pkts", "src_ip_bytes", "dst_pkts", "dst_ip_bytes"]
FEATURES = NOMINAL + NUMERIC

expected_columns = ["row_id"] + FEATURES + ["attack_code"]
assert list(dev.columns) == expected_columns, "Unexpected cyber schema"

X = dev[FEATURES].copy()
y = dev["attack_code"].copy()
for column in ["row_id", "feature_group", "config_group", "fold_id", "attack_code"]:
    assert column not in X.columns, "Forbidden predictor: " + column

assert X.shape[1] == 11, "Expected 11 source predictors"
assert not dev.isna().any().any(), "Missing values in development data"
assert np.isfinite(X[NUMERIC].to_numpy()).all(), "Non-finite numeric predictors"
assert pd.api.types.is_integer_dtype(y), "Class codes must be integers"
assert sorted(y.unique()) == sorted(CLASS_NAMES), "Class codes do not match JSON"

print("Development table:", dev.shape)
print("Predictors:", X.shape, "(", len(NOMINAL), "nominal and", len(NUMERIC), "numeric )")
print("Class names:", CLASS_NAMES)

In [ ]:
# Check that the three tables are aligned before using positions.
assert list(cyber_groups.columns) == ["row_id", "feature_group"], "Unexpected group schema"
assert list(cyber_folds.columns) == ["row_id", "fold_id"], "Unexpected fold schema"
assert dev["row_id"].is_unique, "Duplicate development row_id"
assert dev["row_id"].equals(cyber_groups["row_id"]), "Group rows are not aligned"
assert dev["row_id"].equals(cyber_folds["row_id"]), "Fold rows are not aligned"
assert sorted(cyber_folds["fold_id"].unique()) == [0, 1, 2, 3, 4], "Expected five folds"

# Read the official assignment: the folds are never rebuilt.
folds = []
fold_rows = []
for fold_id in range(5):
    train_idx = np.flatnonzero(cyber_folds["fold_id"].to_numpy() != fold_id)
    val_idx = np.flatnonzero(cyber_folds["fold_id"].to_numpy() == fold_id)

    # A group never appears on both sides of the same split.
    train_groups = set(cyber_groups.iloc[train_idx]["feature_group"])
    val_groups = set(cyber_groups.iloc[val_idx]["feature_group"])
    assert train_groups.isdisjoint(val_groups), "Group leakage in fold " + str(fold_id)
    assert len(train_idx) + len(val_idx) == len(dev), "Incomplete partition"

    folds.append((train_idx, val_idx))
    fold_rows.append({"fold_id": fold_id, "training_rows": len(train_idx),
                      "validation_rows": len(val_idx)})

print("Feature groups:", cyber_groups["feature_group"].nunique())
print(pd.DataFrame(fold_rows).to_string(index=False))
print("All five canonical folds passed the group separation checks.")

The cyber development table has 34,928 rows and 11 source predictors (3 nominal and 8 numeric). The target is `attack_code`, with the 10 class names supplied in the JSON file. There are 34,883 feature groups. The five canonical folds are read from `../folds/cyber_folds.csv`, never rebuilt, and the checks above confirm that no group appears on both sides of a split.

Identifiers, grouping metadata, fold assignments and the target are kept outside `X`. Nothing has been fitted in this block.

## 3. Representation for the tree (statement, Section 2)

**Scaling.** The tree gets no scaler and no `log1p`. A tree asks questions of the form "is `x <= t`?". Any strictly increasing transformation of a numeric column, such as standardisation or `log1p`, changes the threshold `t` but not which rows fall on each side of it, so scaling adds nothing for a tree. Dropping `log1p` also keeps every threshold in the original units (bytes, packets, seconds), so R2 can read a threshold on `dst_bytes` directly in bytes, without converting it back with `e^x - 1`. The eight numeric predictors enter the model unchanged (`"passthrough"`).

**Categorical encoding.** `OneHotEncoder(handle_unknown="ignore", sparse_output=False)` with no level removed. P3 removed one reference level per predictor because the linear models have an intercept. A tree has no intercept and needs no reference level, and removing one would hide that level: with `tcp` removed, the tree could not ask "is the protocol tcp?".

**Unseen levels.** With `handle_unknown="ignore"`, a level that training never saw is encoded as zeros for its predictor. In P3 this coincided with the encoding of the dropped reference level. In P4 no level is dropped, so all zeros now means "none of the known levels". This happens in the data: `service=gssapi` is absent from the training rows of fold 3.

### Correspondence with P3

In [ ]:
# Levels of each nominal predictor in the development table, and the P3 mapping.
# P3 dropped one reference level per predictor (the most frequent one).
P3_DROPPED_LEVELS = ["tcp", "-", "SF"]

mapping_rows = []
for column, dropped in zip(NOMINAL, P3_DROPPED_LEVELS):
    levels = sorted(X[column].unique())
    assert dropped in levels, "P3 reference level not found: " + dropped
    mapping_rows.append({"predictor": column, "levels": len(levels),
                         "P3_dropped_level": dropped,
                         "P3_indicator_columns": len(levels) - 1,
                         "P4_indicator_columns": len(levels)})

mapping = pd.DataFrame(mapping_rows)
print(mapping.to_string(index=False))
print("Total indicator columns, P3:", mapping["P3_indicator_columns"].sum())
print("Total indicator columns, P4:", mapping["P4_indicator_columns"].sum())

P3 dropped `proto=tcp`, `service=-` and `conn_state=SF`, and it standardised the log-transformed numeric predictors. P4 keeps those three levels as the columns `proto_tcp`, `service_-` and `conn_state_SF`, and applies neither `log1p` nor a scaler. The three predictors have 3, 9 and 13 levels, so the indicator columns go from 22 in P3 to 25 in P4. The check below confirms, on every fold, that each P3 indicator column has exactly the same values as the P4 column of the same level; in fold 3 the counts are 21 and 24 because `gssapi` is missing from its training vocabulary.

In [ ]:
def make_preprocessor():
    # One-hot with every level kept: a tree needs no reference level.
    # "ignore" encodes a level unseen in training as all zeros.
    nominal_steps = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

    # Numeric predictors pass through unchanged: no log1p and no scaler.
    preprocessor = ColumnTransformer([
        ("cat", nominal_steps, NOMINAL),
        ("num", "passthrough", NUMERIC),
    ])
    # Returned unfitted so that R2 puts it inside its Pipeline.
    return preprocessor

In [ ]:
# Check the block inside each canonical fold: fitted on training rows only.
check_rows = []
for fold_id, (train_idx, val_idx) in enumerate(folds):
    preprocessor = make_preprocessor()
    assert not hasattr(preprocessor, "transformers_"), "Expected an unfitted transformer"
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[val_idx]
    X_train_t = preprocessor.fit_transform(X_train)
    X_valid_t = preprocessor.transform(X_valid)

    encoder = preprocessor.named_transformers_["cat"]
    assert encoder.drop_idx_ is None, "No level may be dropped"

    # Count the columns and the validation levels that training never saw.
    n_cat = 0
    unseen_per_row = np.zeros(len(X_valid), dtype=int)
    for column, categories in zip(NOMINAL, encoder.categories_):
        assert list(categories) == sorted(X_train[column].unique()), "Wrong vocabulary"
        n_cat = n_cat + len(categories)
        unseen_per_row = unseen_per_row + (~X_valid[column].isin(categories)).to_numpy()
    n_num = len(NUMERIC)

    feature_names = preprocessor.get_feature_names_out()
    assert X_train_t.shape[1] == len(feature_names), "Missing feature names"
    assert X_valid_t.shape[1] == len(feature_names), "Inconsistent columns"
    assert len(feature_names) == n_cat + n_num, "Unexpected number of columns"
    assert np.isfinite(X_train_t).all(), "Non-finite training values"
    assert np.isfinite(X_valid_t).all(), "Non-finite validation values"

    # Numeric columns keep their original values: nothing was scaled.
    assert np.array_equal(X_train_t[:, n_cat:], X_train[NUMERIC].to_numpy()), "Train values changed"
    assert np.array_equal(X_valid_t[:, n_cat:], X_valid[NUMERIC].to_numpy()), "Valid values changed"

    # One indicator per predictor is 1 in training; an unseen level gives 0.
    train_ones = X_train_t[:, :n_cat].sum(axis=1)
    valid_ones = X_valid_t[:, :n_cat].sum(axis=1)
    assert (train_ones == len(NOMINAL)).all(), "Each training row needs one level per predictor"
    assert (valid_ones == len(NOMINAL) - unseen_per_row).all(), "Unseen levels must give zeros"

    if fold_id == 0:
        names_fold_0 = list(feature_names)

    check_rows.append({"fold_id": fold_id, "output_features": len(feature_names),
                       "indicator_columns": n_cat, "numeric_columns": n_num,
                       "unseen_validation_rows": int((unseen_per_row > 0).sum())})

print(pd.DataFrame(check_rows).to_string(index=False))
print("Fold 0 output names:", names_fold_0)
print("Preprocessing checks passed on all five folds.")

In [ ]:
# Every P3 indicator column must equal the P4 column of the same level.
for fold_id, (train_idx, val_idx) in enumerate(folds):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[val_idx]

    p4 = make_preprocessor()
    p4.fit(X_train)
    p4_names = list(p4.named_transformers_["cat"].get_feature_names_out(NOMINAL))
    p4_valid = p4.named_transformers_["cat"].transform(X_valid[NOMINAL])

    p3 = OneHotEncoder(drop=P3_DROPPED_LEVELS, handle_unknown="ignore", sparse_output=False)
    p3.fit(X_train[NOMINAL])
    p3_names = list(p3.get_feature_names_out(NOMINAL))
    p3_valid = p3.transform(X_valid[NOMINAL])

    for position, name in enumerate(p3_names):
        assert name in p4_names, "P3 column missing in P4: " + name
        same_column = p4_valid[:, p4_names.index(name)]
        assert np.array_equal(p3_valid[:, position], same_column), "Different values: " + name
    print("Fold", fold_id, ": P3 columns", len(p3_names), "are all in P4 columns", len(p4_names))

In [ ]:
# The real unseen level of the statement: service=gssapi in fold 3.
train_idx, val_idx = folds[3]
preprocessor = make_preprocessor()
preprocessor.fit(X.iloc[train_idx])
encoder = preprocessor.named_transformers_["cat"]
assert "gssapi" not in encoder.categories_[1], "gssapi should be unseen in fold 3"

validation_rows = X.iloc[val_idx]
unseen_row = validation_rows.loc[validation_rows["service"] == "gssapi", NOMINAL]
print("Validation rows with service=gssapi in fold 3:", len(unseen_row))

# Positions of the service block inside the encoded row.
start = len(encoder.categories_[0])
stop = start + len(encoder.categories_[1])
encoded = encoder.transform(unseen_row)
print("Service indicators set to 1 for that row:", int(encoded[0, start:stop].sum()))
assert encoded[:, start:stop].sum() == 0, "An unseen service must be all zeros"

**Checks.** All checks pass on the five canonical folds. The transformed matrix has 33 columns in folds 0, 1, 2 and 4 (25 indicators and 8 numeric) and 32 in fold 3 (24 indicators and 8 numeric), because the training rows of fold 3 contain no `service=gssapi`. Within each fold, training and validation have the same columns, the numeric columns keep their original values, and every training row has exactly one indicator set per nominal predictor.

The only validation row with an unseen level is the `service=gssapi` row of fold 3. Its service block is all zeros, as the last check shows. scikit-learn prints an unknown-category warning the first time it meets that row: this is expected, because the encoder applies the declared all-zero fallback. These are preprocessing checks, not model results.

`make_preprocessor()` returns a new, unfitted transformer. R2 must place it inside the model `Pipeline`, so that the encoder vocabulary is learned only from each fold's training rows. The output names follow the pattern shown above (`cat__service_http`, `num__dst_bytes`), and they differ between folds, so importances and rules must be read with each fitted preprocessor's `get_feature_names_out()`. To go back to the 11 original variables, sum the columns of each nominal predictor (the names that start with `cat__proto_`, `cat__service_` or `cat__conn_state_`).

## 4. Canonical folds and metrics

In [ ]:
fold_table = pd.read_csv("../folds/cyber_folds.csv")
#the positions only make sense if the fold file lists the rows in the same order as dev
assert np.array_equal(fold_table["row_id"].to_numpy(), dev["row_id"].to_numpy())
fold_id = fold_table["fold_id"].to_numpy()

#lets build the folds
folds = []
for f in range(5):
    train_idx = np.where(fold_id != f)[0]
    val_idx = np.where(fold_id == f)[0]
    folds.append((train_idx,val_idx))

print(len(folds))



In [ ]:
grps = cyber_groups["feature_group"].to_numpy()
#the same for the groups file
assert np.array_equal(cyber_groups["row_id"].to_numpy(), dev["row_id"].to_numpy())
#we check that the patterns are in different groups in train and in validation
for train_idx, val_idx in folds:
    grps_train = grps[train_idx]
    grps_train = set(grps_train)
    grps_val = grps[val_idx]
    grps_val = set(grps_val) #we created two different sets to see the distinct groups in each one
    assert(len(grps_train & grps_val) == 0) #here we check that the intersection between the two groups is 0
    print(len(train_idx),len(val_idx))

print("no feature group is in both training and validation")




In [ ]:
def spread(values):
    res = np.max(values) - np.min(values)
    
    return res

def cyber_metrics(y_true, y_pred):
    macro_f1 = metrics.f1_score(y_true,y_pred,average="macro")
    weighted_f1 = metrics.f1_score(y_true,y_pred,average="weighted")
    accuracy = metrics.accuracy_score(y_true,y_pred)


    return [macro_f1, weighted_f1, accuracy]


In [ ]:
#check: the spread of the P2 KNN must be 0.0062, as in comparativa.csv
print(round(spread([0.9436, 0.9473, 0.9490, 0.9428, 0.9451]), 4))

# P4.1 — Tree classification

In [ ]:
def make_tree():
    return Pipeline([
        ("preprocessing", make_preprocessor()),
        ("tree", DecisionTreeClassifier(random_state=PROJECT_RANDOM_STATE)),
    ])

param_grid = {
    "tree__max_depth": [3, 5, 8, 12, 20, None],
    "tree__min_samples_leaf": [1, 5, 20, 50],
    "tree__ccp_alpha": [0.0, 1e-5, 1e-4, 1e-3],
    "tree__criterion": ["gini", "entropy"],
}
n_candidates = 1
for values in param_grid.values():
    n_candidates = n_candidates * len(values)
print("Declared grid:", param_grid)
print("Candidates:", n_candidates)
print("Cross-validation fits:", n_candidates * len(folds))

**Declared grid and why.** We try shallow and deeper trees, including
unlimited depth, to see whether extra splits help validation. The minimum leaf
sizes range from individual rows to larger leaves, which can reduce overfitting.
An alpha of zero means no pruning; the positive values try increasing amounts of
pruning. We also compare Gini and entropy. Their impurity scales differ, so the
same alpha does not necessarily prune them equally.

We select the configuration with the highest mean validation macro F1 on the
canonical folds. The seed stays fixed, and an exact score tie is resolved by the
search order. As required by the handout, we use these same folds for selection
and reporting, without an extra holdout. This can make the selected score
optimistic, so it should not be treated as an independent test result.

In [ ]:
search = GridSearchCV(
    make_tree(),
    param_grid,
    cv=folds,
    scoring="f1_macro",
    return_train_score=True,
    n_jobs=4,
    error_score="raise",
)
start = time.perf_counter()
search.fit(X, y)
search_seconds = time.perf_counter() - start
print("Search time including final refit (s):", round(search_seconds, 3))

In [ ]:
best = search.best_params_
best_index = search.best_index_
for parameter in param_grid:
    print(parameter, "=", best[parameter], "| tried:", param_grid[parameter])
print("Selected mean validation macro F1:", round(search.best_score_, 6))

grid_results = pd.DataFrame(search.cv_results_)

In [ ]:
# GridSearchCV refits the winner on all development rows.
final_pipe = search.best_estimator_
final_tree = final_pipe.named_steps["tree"]
feature_names = final_pipe.named_steps["preprocessing"].get_feature_names_out()
t = final_tree.tree_
print("Description of best_estimator_, fitted on ALL development rows")
print("Training rows at root:", t.n_node_samples[0])
print("Nodes:", t.node_count)
print("Depth:", final_tree.get_depth())
print("Leaves:", final_tree.get_n_leaves())
if t.children_left[0] != -1:
    print("Root feature:", feature_names[t.feature[0]])
    print("Root threshold:", t.threshold[0])
else:
    print("The root is a leaf; there is no split.")
print("This is a fitted-object description, not validation performance.")

In [ ]:
# Refit the chosen configuration on each training fold, then predict validation.
fold_rows = []
oof_prediction = np.full(len(y), -1, dtype=int)
oof_coverage = np.zeros(len(y), dtype=int)
for f, (train_idx, val_idx) in enumerate(folds):
    model = make_tree()
    model.set_params(**best)
    start = time.perf_counter()
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    fit_seconds = time.perf_counter() - start
    prediction = model.predict(X.iloc[val_idx])
    scores = cyber_metrics(y.iloc[val_idx], prediction)
    grid_score = search.cv_results_["split" + str(f) + "_test_score"][best_index]
    assert np.isclose(scores[0], grid_score, atol=1e-12, rtol=0)
    oof_prediction[val_idx] = prediction
    oof_coverage[val_idx] = oof_coverage[val_idx] + 1
    fold_rows.append({"fold": f, "macro_f1": scores[0],
                      "weighted_f1": scores[1], "accuracy": scores[2],
                      "fit_time_s": fit_seconds})

assert (oof_coverage == 1).all()
assert (oof_prediction >= 0).all()
fold_metrics = pd.DataFrame(fold_rows)
tree_cyber = fold_metrics["macro_f1"].to_numpy()
assert np.isclose(tree_cyber.mean(), search.best_score_, atol=1e-12, rtol=0)
print(fold_metrics.round(6).to_string(index=False))
print("Mean macro F1:", round(tree_cyber.mean(), 6))
print("Macro F1 range (max - min):", round(spread(tree_cyber), 6))
print("Mean weighted F1:", round(fold_metrics["weighted_f1"].mean(), 6))
print("Mean accuracy:", round(fold_metrics["accuracy"].mean(), 6))
print("Mean fit time for one pipeline (s):", round(fold_metrics["fit_time_s"].mean(), 6))

In [ ]:
CLASS_CODES = sorted(CLASS_NAMES)
class_names = []
for class_code in CLASS_CODES:
    class_names.append(CLASS_NAMES[class_code])

report = metrics.classification_report(
    y, oof_prediction, labels=CLASS_CODES, target_names=class_names,
    output_dict=True, zero_division=0,
)
class_report = pd.DataFrame(report).T.loc[class_names]
print("Per-class metrics from pooled out-of-fold predictions:")
print(class_report.round(4).to_string())
print("Pooled OOF accuracy:", round(metrics.accuracy_score(y, oof_prediction), 6))
print("Pooled OOF macro F1:",
      round(metrics.f1_score(y, oof_prediction, average="macro"), 6))

cm = metrics.confusion_matrix(y, oof_prediction, labels=CLASS_CODES)
assert cm.sum() == len(y)
assert np.array_equal(cm.sum(axis=1), class_report["support"].to_numpy())
print("Rows = true class; columns = predicted class")
print(pd.DataFrame(cm, index=class_names, columns=class_names).to_string())
fig, ax = plt.subplots(figsize=(12, 10))
display = metrics.ConfusionMatrixDisplay(cm, display_labels=class_names)
display.plot(ax=ax, xticks_rotation=45, colorbar=False, values_format="d")
ax.set_title("Selected tree: pooled out-of-fold confusion matrix")
fig.tight_layout()
plt.show()

**Interpretation.** The search selects `criterion="entropy"`,
`max_depth=20`, `min_samples_leaf=1` and `ccp_alpha=0.0001`. Mean validation macro
F1 is **0.9457**, ranging from 0.9415 to 0.9499 across folds, with a range of
0.0083. Mean weighted F1 is 0.9625 and mean accuracy is 0.9622. The higher weighted
F1 shows that the larger classes have better results overall than the classes
given equal weight by macro F1.

The weakest class is `mitm`, with pooled F1 0.7656 and recall 0.8075 on 805 rows.
The rarest class, `ransomware`, has F1 0.9356 on 208 rows. Its score is good here,
but it is based on fewer examples. These class scores and the confusion matrix
use validation predictions only. Pooling predictions and averaging fold scores
are different calculations, so their macro F1 values need not match exactly.

The tree refitted on all development rows has 1235 nodes, depth 20 and 618 leaves.
These describe that fit, not the separate fold models. Individual paths can be
read, but the whole tree is too large to explain as a short list of rules.

**Execution environment.** The versions used for this run are printed above.
They come from the available Anaconda environment and differ from the versions
in `requirements.txt`. The team should agree on the environment for the final
run, since small version differences can affect the results.

# P4.2 — The complexity study

For each curve, we change one hyperparameter and keep the others at
their selected values. Both lines show macro F1 averaged over the same canonical
folds, with preprocessing fitted inside each training fold. These curves help us
understand complexity; they do not start a new hyperparameter search.

For the selected configuration, mean training F1 is 0.9797 and mean validation
F1 is 0.9457. The gap shows that the model fits training rows more closely than
unseen rows. We use training scores only as a diagnostic in this section.

In [ ]:
# Training scores are used here only to diagnose model complexity.
print("Selected mean training macro F1:",
      round(search.cv_results_["mean_train_score"][best_index], 6))
print("Selected mean validation macro F1:", round(search.best_score_, 6))

columns = ["rank_test_score", "param_tree__max_depth", "param_tree__min_samples_leaf",
           "param_tree__ccp_alpha", "param_tree__criterion", "mean_train_score",
           "mean_test_score", "mean_fit_time"]
print(grid_results.sort_values("rank_test_score")[columns].head(10).to_string(index=False))

In [ ]:
curve_values = {
    "tree__max_depth": [2, 3, 5, 8, 12, 20, None],
    "tree__ccp_alpha": [0.0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1],
}
curve_tables = {}
for parameter, values in curve_values.items():
    curve_model = make_tree()
    curve_model.set_params(**best)
    train_scores, val_scores = validation_curve(
        curve_model, X, y, param_name=parameter, param_range=values,
        cv=folds, scoring="f1_macro", n_jobs=4, error_score="raise",
    )
    curve_table = pd.DataFrame({
        "value": values,
        "train_mean": train_scores.mean(axis=1),
        "validation_mean": val_scores.mean(axis=1),
        "train_range": np.ptp(train_scores, axis=1),
        "validation_range": np.ptp(val_scores, axis=1),
    })
    curve_tables[parameter] = curve_table
    print(parameter)
    print(curve_table.round(6).to_string(index=False))

In [ ]:
for parameter, values in curve_values.items():
    curve_table = curve_tables[parameter]
    positions = np.arange(len(values))
    labels = []
    for value in values:
        labels.append(str(value))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(positions, curve_table["train_mean"], "o-", label="Training mean")
    ax.plot(positions, curve_table["validation_mean"], "s-", label="Validation mean")
    ax.set_xticks(positions, labels)
    ax.set_xlabel(parameter + " (declared values; equally spaced positions)")
    ax.set_ylabel("Macro F1")
    ax.set_title("Complexity slice: other parameters fixed at the selected values")
    if parameter == "tree__max_depth":
        left_label = "Lower depth: underfitting risk"
        right_label = "Higher depth: overfitting risk"
    else:
        left_label = "Less pruning: overfitting risk"
        right_label = "More pruning: underfitting risk"
    ax.text(0.02, 0.04, left_label, transform=ax.transAxes, fontsize=9)
    ax.text(0.98, 0.04, right_label, transform=ax.transAxes, ha="right", fontsize=9)
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()

**Reading of the curves.**

- **Depth:** on the left, depth 2 gives training F1 0.2535 and validation F1
  0.2530. Both are low, so the tree is too simple and underfits. On the right,
  unlimited depth gives training F1 0.9814 and validation F1 0.9455. Training keeps
  improving more than validation, so this side has more risk of overfitting.
- **Pruning:** the direction is reversed. On the left, alpha 0 gives training
  F1 0.9916 and validation F1 0.9442, leaving a clear gap. On the right, alpha
  0.1 reduces them to 0.4326 and 0.4316. This strong pruning removes useful splits
  and underfits. A small gap is therefore not enough: both scores can be poor.

The other settings stay fixed in each curve. These results do not mean that every
deep tree overfits or that pruning always helps. The ranges in the tables describe
variation across folds; they are not confidence intervals.

In the depth table, `NaN` represents `max_depth=None` (no depth limit). The score columns still contain valid results.

In [ ]:
# A maximal tree's path on the TRAINING portion of canonical fold 0 only.
path_fold = 0
train_idx, val_idx = folds[path_fold]
path_preprocessor = make_preprocessor()
X_train_t = path_preprocessor.fit_transform(X.iloc[train_idx])
path_tree = DecisionTreeClassifier(
    criterion=best["tree__criterion"], random_state=PROJECT_RANDOM_STATE,
)
path = path_tree.cost_complexity_pruning_path(X_train_t, y.iloc[train_idx])
path_table = pd.DataFrame({"ccp_alpha": path.ccp_alphas,
                           "total_leaf_impurity": path.impurities})
assert np.all(np.diff(path.ccp_alphas) >= -1e-12)
assert np.all(np.diff(path.impurities) >= -1e-12)
print("Path fold:", path_fold, "| training rows:", len(train_idx))
print("Criterion:", best["tree__criterion"])
print("Path settings: max_depth=None, min_samples_leaf=1, ccp_alpha=0")
print("Number of alpha entries:", len(path_table))
print("Number of distinct alphas:", len(np.unique(path.ccp_alphas)))
print("Alpha minimum:", path.ccp_alphas.min())
print("Alpha maximum:", path.ccp_alphas.max())
print(path_table.head().to_string(index=False))
print(path_table.tail().to_string(index=False))
fig, ax = plt.subplots(figsize=(8, 5))
ax.step(path.ccp_alphas, path.impurities, where="post")
ax.set_xlabel("ccp_alpha (selected criterion's impurity units)")
ax.set_ylabel("Total sample-weighted leaf impurity")
ax.set_title("Cost-complexity path: canonical fold 0 training rows only")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

**What the pruning path shows.** We use only the training rows of fold 0
and grow an unrestricted tree with the selected entropy criterion. Its path has
778 alpha entries, including repeated values, and 544 distinct values. The range
is from 0 to 0.499668. Increasing alpha removes subtrees and increases training
leaf impurity; the last step leaves only the root.

Here alpha is measured in entropy units. It cannot be read as the same amount of
pruning as an alpha from a squared-error regression tree. As explained in Topic 8,
scikit-learn uses weighted leaf impurity for pruning, rather than classification
error. The path gives candidate alphas and shows how pruning works; it does not
choose the best one. Our model is selected using validation macro F1.

# P4.4 — What a tree importance says

In [ ]:
# The selected tree refitted on all development rows, as in the size description.
print("Importances from best_estimator_, fitted on", t.n_node_samples[0], "rows")
importance_rows = []
for column in FEATURES:
    total = 0.0
    for position, name in enumerate(feature_names):
        if column in NOMINAL:
            belongs = name.startswith("cat__" + column + "_")
        else:
            belongs = name == "num__" + column
        if belongs:
            total = total + final_tree.feature_importances_[position]
    importance_rows.append({"predictor": column, "importance": total})

importances = pd.DataFrame(importance_rows)
importances = importances.sort_values("importance", ascending=False)
assert np.isclose(importances["importance"].sum(), 1.0)
print(importances.round(6).to_string(index=False))
print("Importance sum:", importances["importance"].sum())

In [ ]:
plot_importances = importances.sort_values("importance")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(plot_importances["predictor"], plot_importances["importance"])
ax.set_xlabel("Normalised weighted impurity decrease")
ax.set_title("Selected tree refitted on all development rows: source importances")
fig.tight_layout()
plt.show()

**What a tree importance means.** It is the share of the total weighted
impurity decrease due to splits on that predictor. It is calculated on the
training rows of the stated fit, across the whole tree and all classes. There is
one importance per predictor, not one per class, and the shares sum to one.

We use the selected tree refitted on all development rows and sum each nominal
predictor's one-hot columns. The largest source importances are `service`
(0.1945), `src_ip_bytes` (0.1781) and `conn_state` (0.1585).

**What a tree importance does not say.** It has no sign: it does not tell us which
class a predictor pushes towards, whether larger values favour an attack, or how
much a prediction would change if the predictor changed. It is not a measure of
performance on new rows or evidence of a causal effect. Correlated predictors
can share importance, and predictors with more possible splits can be favoured.

# P4.5 — Challenge A against Practice 3 and Practice 2

In [ ]:
# five per-fold primary scores of one earlier row of comparativa.csv
def previous_scores(milestone, challenge, family):
    table = pd.read_csv("../comparativa.csv")
    # keep only the row with this milestone, challenge and family
    row = table[(table["milestone"] == milestone) & (table["challenge"] == challenge) & (table["family"] == family)]
    # the five scores are stored as one text, e.g. "0.9436 0.9473 0.9490 0.9428 0.9451"
    text = row["primary_metric_per_fold"].iloc[0]
    # cut the text at the spaces and turn each piece into a number
    return np.array(text.split(), dtype=float)




p2_cyber = previous_scores("P2", "cyber", "KNN")
p3_cyber = previous_scores("P3", "cyber", "LogisticRegression")
print("P2 KNN:   ", p2_cyber)
print("P3 logreg:", p3_cyber)

# check of the method with rows we already have: P3 - P2 must give
# the differences printed in Practice 3 (mean -0.1074)
deltas = p3_cyber - p2_cyber
print("P3 - P2:  ", np.round(deltas, 4))
print("mean:", round(np.mean(deltas), 4), "| spread:", round(spread(deltas), 4))

# R2's five macro F1 of the selected tree (from R2's notebook, printed there with 6 decimals)
# delta = tree - previous: a positive delta means the tree did better on that fold
tree_cyber = np.array([0.945324, 0.949640, 0.949863, 0.941546, 0.942337])

p3_diff = tree_cyber - p3_cyber
p2_diff = tree_cyber - p2_cyber

print("tree:     ", np.round(tree_cyber, 4), "| mean:", round(np.mean(tree_cyber), 4), "| spread:", round(spread(tree_cyber), 4))
print()
print("tree - P3:", np.round(p3_diff, 4))
print("mean:", round(np.mean(p3_diff), 4), "| spread:", round(spread(p3_diff), 4), "| folds where the tree is better:", np.sum(p3_diff > 0), "of 5")
print("spread of P3 logreg:", round(spread(p3_cyber), 4))
print()
print("tree - P2:", np.round(p2_diff, 4))
print("mean:", round(np.mean(p2_diff), 4), "| spread:", round(spread(p2_diff), 4), "| folds where the tree is better:", np.sum(p2_diff > 0), "of 5")
print("spread of P2 KNN:", round(spread(p2_cyber), 4))

**Reading of the differences, and which kind of comparison this is.**

*Against the Practice 3 logistic model.* The tree is better on all five folds. The mean paired difference is +0.1076 in macro F1, with a range (max − min) of 0.0119 across the five differences. The fold-to-fold range of each model is small next to this change (0.0083 for the tree and 0.0121 for the logistic model), so the change is large relative to the variability we observed.

*Against the Practice 2 KNN.* The two models are practically level. The mean paired difference is +0.0002, the tree is better on three folds and worse on two, and the differences go from −0.0028 to +0.0023. This change is small next to the fold-to-fold range of either model (0.0083 for the tree and 0.0062 for the KNN).

Both readings describe these five folds only. They are not a significance test, and they do not establish that a difference would hold on new data.

*Which kind of comparison.* Both are **pipeline comparisons**, not isolated family comparisons, because the representation changed. The tree keeps every one-hot level and uses the numeric predictors unchanged. The Practice 3 model dropped one reference level per nominal predictor and used `log1p` and `StandardScaler`. The Practice 2 KNN kept every level, like the tree, but also used `log1p` and `StandardScaler`. For a tree, `log1p` and scaling are increasing transformations: they move each threshold without changing which rows fall on each side, so they would not change the tree's partitions. Even so, we did not rerun the earlier families under the tree's representation, so we only claim that this pipeline did better than (P3) or about the same as (P2) the earlier pipelines on these folds.

In [ ]:
# Historical fit times are snapshots, not a controlled timing benchmark.
comparison_table = pd.read_csv("../comparativa.csv")
cyber_history = comparison_table[comparison_table["challenge"] == "cyber"]
print(cyber_history[["milestone", "family", "route", "primary_metric",
                     "primary_metric_spread", "fit_time_s"]].to_string(index=False))
print("Current tree search time (s):", round(search_seconds, 3))
print("Current tree mean single-pipeline fit time (s):",
      round(fold_metrics["fit_time_s"].mean(), 6))
print("Current selected-tree leaves:", final_tree.get_n_leaves())

dummy_cyber = previous_scores("P2", "cyber", "Dummy").mean()
dummy_storage = previous_scores("P2", "storage", "Dummy").mean()
gap_rows = []
for milestone, cyber_scores, storage_family in [
    ("P2", p2_cyber, "KNN"), ("P3", p3_cyber, "LinearRegression"),
]:
    storage_scores = previous_scores(milestone, "storage", storage_family)
    gap_rows.append({"milestone": milestone,
                     "cyber_gap_closed": (cyber_scores.mean() - dummy_cyber)
                                         / (1 - dummy_cyber),
                     "storage_dummy_MAE_removed": 1 - storage_scores.mean() / dummy_storage})
print("Relative-to-Dummy summaries (different tasks, not a common difficulty scale):")
print(pd.DataFrame(gap_rows).round(4).to_string(index=False))
print("P4 cyber gap closed:",
      round((tree_cyber.mean() - dummy_cyber) / (1 - dummy_cyber), 4))

### Storage results used for context

The storage scores below come from R3's executed draft at
`notebooks/borradores/R3_p4_storage.ipynb`, commit `7f79f315` on `r3-storage`.
They are a saved snapshot, not a regression experiment carried out in R2.

The latest checked role notebook, commit `1f8f94f`, still has no executed
results. We therefore keep this comparison provisional until R3/R4 confirm the
storage scores for the final submission.

In [ ]:
# R3's saved draft output, rounded to the four decimals it prints.
r3_draft_storage = np.array([0.1238, 0.1286, 0.1319, 0.1353, 0.1332])
print("R3 draft storage MAE(z), folds 0 to 4:", r3_draft_storage)
print("R3 draft storage mean MAE(z):", round(r3_draft_storage.mean(), 4))
print("R3 draft share of Dummy MAE removed:",
      round(1 - r3_draft_storage.mean() / dummy_storage, 4))
print("Cyber share of gap to perfect F1 closed:",
      round((tree_cyber.mean() - dummy_cyber) / (1 - dummy_cyber), 4))

### The six fixed comparison questions

1. **Size and consistency of the change.** Against P3 logistic regression, the
   tree improves macro F1 on all five folds. The mean difference is +0.1076, from
   +0.1046 to +0.1164, with a range of 0.0119. This improvement is much larger than
   the fold ranges of the tree (0.0083) and logistic regression (0.0121), so the
   change is large and consistent in these folds.

   Against P2 KNN, the mean difference is only +0.0002. The tree wins in three
   folds and loses in two; differences go from -0.0028 to +0.0023, with a range
   of 0.0051. The mean change is much smaller than the fold ranges of the tree
   (0.0083) and KNN (0.0062), so there is no clear practical improvement here.
   These are descriptive comparisons, not statistical tests. We compare at
   the four-decimal precision of the historical scores. Both the tree and KNN
   were tuned on these folds, so their selected scores may be optimistic.

2. **Route and representation.** All three models use the primary route, the same
   source predictors and the same canonical folds. P3 drops `proto=tcp`,
   `service=-` and `conn_state=SF`, and standardises the log-transformed numeric
   values. P2 KNN keeps all one-hot levels but also uses `log1p` and scaling.
   Our tree keeps every level and uses raw numeric values.

   Both comparisons are therefore pipeline comparisons. Scaling adds no useful
   split options to a tree, but it changes the distances used by KNN. The results
   compare these particular pipelines; they do not prove that one model family
   is always better than another.

3. **What can the new family express?** Unlike the additive class scores of the
   logistic model, a tree can use different conditions in different parts of the
   data and combine predictors along a path. This gives nonlinear boundaries and
   interactions. KNN can also produce nonlinear boundaries, so the tree is not
   automatically more expressive than KNN. The main difference is that it uses
   threshold rules instead of neighbour votes.

4. **What did it cost?** The search tried 192 configurations, requiring 960 fold
   fits, and took 53.611 seconds including the final refit. A selected pipeline
   fit took 0.152849 seconds on average. Historical fit times are printed above,
   but they were measured in separate runs, so they are only rough references.
   This search takes more tuning than P3's fixed logistic model and tries more
   settings than P2's neighbour-count search. A tree path is readable, but 618
   leaves make the whole tree difficult to explain. Logistic regression offers
   coefficients and KNN offers neighbours as different ways to inspect a result.

5. **Did the ordering of the challenges' difficulty change?** We cannot compare
   cyber F1 directly with storage MAE because they measure different things.
   Relative to each Dummy baseline, cyber closes more of its gap to perfect F1
   than storage removes of its Dummy MAE in P2 and P3. Using R3's draft, the same
   pattern remains in P4: the printed shares are 0.9444 for cyber and 0.8071 for
   storage. This suggests no reversal under this comparison, but it is not a
   common difficulty scale. The tree can help with nonlinear patterns in both
   tasks, while storage still has to predict latency for unseen configurations.
   This answer remains provisional until R3/R4 confirm the storage results.

6. **Would you ship the new model?** I would keep both the tree and KNN as
   candidates. The tree clearly improves on the P3 logistic model on these folds,
   but its small average advantage over KNN is not enough to justify replacing
   KNN. I would first check performance on independent data under the course
   protocol, errors on costly attack classes, prediction time and memory use.
   Better operational results, or a simpler tree with similar performance, could
   make me choose it. The development scores alone do not settle that decision.

## Comparison table

In [ ]:
# the Practice 4 cyber row, appended at the end of comparativa.csv
# the P2 and P3 rows are never rewritten: mode="a" only adds lines at the end of the file
table = pd.read_csv("../comparativa.csv")

cyber_row = {
    "milestone": "P4", "challenge": "cyber", "family": "DecisionTree", "route": "primary",
    "configuration": "criterion=entropy, max_depth=20, min_samples_leaf=1, ccp_alpha=0.0001; selected by highest mean macro F1 "
                     "from max_depth [3, 5, 8, 12, 20, None], min_samples_leaf [1, 5, 20, 50], ccp_alpha [0, 1e-05, 0.0001, 0.001], criterion [gini, entropy]",
    "preprocessing": "one-hot on the 3 nominal predictors with every level kept; numeric predictors unchanged (no log1p, no scaler); fitted inside each fold",
    "features": "features.md: the 11 distributed source predictors (frozen basis)",
    "validation": "canonical StratifiedGroupKFold(5) on feature_group, read from folds/cyber_folds.csv; grid search with cv=folds",
    "primary_metric_per_fold": " ".join(f"{v:.4f}" for v in tree_cyber),
    "primary_metric": round(np.mean(tree_cyber), 4),
    "primary_metric_spread": round(spread(tree_cyber), 4),
    "supporting_metrics": "weighted F1 0.9625; accuracy 0.9622; per-class precision/recall/F1/support and confusion in p4_cyber.ipynb",
    "fit_time_s": round(fold_metrics["fit_time_s"].mean(), 4),
    "interpretability": "rules from root to leaf; source-level impurity importances (no sign, training fit)",
    "notes": "non-linear model; about equal to the P2 KNN and clearly above the P3 logistic model on every fold; mitm is the weakest class",
}

for row in [cyber_row]:
    # the row must have exactly the columns of the table, in the same order
    assert list(row.keys()) == list(table.columns)
    # if the notebook is run again, do not add the same row twice
    already = (table["milestone"] == row["milestone"]) & (table["challenge"] == row["challenge"]) & (table["family"] == row["family"])
    if already.any():
        print("already in comparativa.csv:", row["milestone"], row["challenge"], row["family"])
    else:
        pd.DataFrame([row]).to_csv("../comparativa.csv", mode="a", header=False, index=False)
        print("added to comparativa.csv:", row["milestone"], row["challenge"], row["family"])

# the whole table after the append: P2 and P3 rows unchanged, P4 rows at the end
final_table = pd.read_csv("../comparativa.csv")
print(final_table[["milestone", "challenge", "family", "route", "primary_metric", "primary_metric_spread"]].to_string(index=False))